# Логирование Градиентного бустинга в MLflow для проекта "Определение популярности геолокации для размещения банкомата"

## 1. Импорты и настройки окружения

In [23]:
import os
import warnings
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import train_test_split, learning_curve, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.preprocessing import StandardScaler


import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

import optuna
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
load_dotenv()

# Цвета для вывода
GREEN = '\033[92m'
BLUE = '\033[94m'
YELLOW = '\033[93m'
RESET = '\033[0m'

## 2. Настройка MLflow и S3

In [24]:
# Для локального стенда из docker-compose
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID", "admin")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY", "password")
raw_s3_endpoint = os.getenv("MLFLOW_S3_ENDPOINT_URL", "http://localhost:9000")

# `minio` резолвится только внутри docker-сети. Для локального ноутбука нужен localhost.
if "minio:9000" in raw_s3_endpoint:
    raw_s3_endpoint = "http://localhost:9000"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = raw_s3_endpoint

# Настройка MLflow
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5050")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)


# Быстрая проверка подключения
mlflow.search_experiments(max_results=3)
print(f"{GREEN}Подключение к MLflow успешно{RESET}")

print("MLflow URI:", mlflow.get_tracking_uri())

Подключение к MLflow успешно
MLflow URI: http://localhost:5050


## 3. Подготовка датасета

In [25]:
df = pd.read_csv('./data/train_with_new_features.csv')
df = df.drop(columns=['id', 'address', 'address_rus'])
df = df.dropna()

df['population'] = df['population'].str.replace("\xa0", "").astype(float)
df['atm_group'] = df['atm_group'].astype('float')

X = df.drop(columns=['target'])
y = df['target']

# Загрузка тестовых данных
df_test = pd.read_csv('./data/test_with_new_features.csv')
df_test = df_test.drop(columns=['Unnamed: 0', 'id', 'address', 'address_rus'])
df_test = df_test.dropna()
df_test['population'] = df_test['population'].astype(str).str.replace("\xa0", "").astype(float)
df_test['atm_group'] = df_test['atm_group'].astype('float')

X_test = df_test.drop(columns=['target'])
y_test = df_test['target']

# Конвертируем колонки, которые должны быть int
int_columns = ['schools_nearby', 'supermarket_nearby', 'mall_nearby', 'bar_nearby', 
               'cafe_nearby', 'restaurant_nearby', 'police_nearby', 'post_office_nearby',
               'place_of_worship_nearby', 'university_nearby', 'cinema_nearby', 
               'casino_nearby', 'nightclub_nearby']

for col in int_columns:
    X_test[col] = X_test[col].astype('int64')

# Убеждаемся, что колонки совпадают
column_names = X.columns
X = X[column_names]
X_test = X_test[column_names]

# Разделение на train/val
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("train:", X_train.shape, "val:", X_val.shape, "test:", X_test.shape)

train: (4988, 18) val: (1248, 18) test: (2498, 18)


## 4. Baseline модели

In [26]:
def train_baseline_with_tuning(X_train, y_train, X_test, y_test):
    
    results = {}
    
    # 1. Ridge Regression (L2)
    print(f"{BLUE}Обучаем Ridge Regression с подбором параметров...{RESET}")
    ridge_params = {'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}
    ridge_grid = GridSearchCV(Ridge(random_state=42), ridge_params, cv=3, scoring='r2', n_jobs=-1)
    ridge_grid.fit(X_train, y_train)
    ridge_best = ridge_grid.best_estimator_
    y_pred_ridge = ridge_best.predict(X_test)
    
    results['ridge'] = {
        'model': ridge_best,
        'best_params': ridge_grid.best_params_,
        'r2': r2_score(y_test, y_pred_ridge),
        'mae': mean_absolute_error(y_test, y_pred_ridge),
        'predictions': y_pred_ridge
    }
    print(f"Лучшие параметры: {ridge_grid.best_params_}")
    print(f"R^2: {results['ridge']['r2']:.4f}")
    
    # 2. Random Forest
    print(f"\n{BLUE}Обучаем Random Forest с подбором параметров...{RESET}")
    rf_params = {
        'n_estimators': [50, 100, 200],
        'max_depth': [5, 10, None],
        'min_samples_split': [2, 5, 10]
    }
    rf_grid = GridSearchCV(RandomForestRegressor(random_state=42), rf_params, cv=3, scoring='r2', n_jobs=-1)
    rf_grid.fit(X_train, y_train)
    rf_best = rf_grid.best_estimator_
    y_pred_rf = rf_best.predict(X_test)
    
    results['random_forest'] = {
        'model': rf_best,
        'best_params': rf_grid.best_params_,
        'r2': r2_score(y_test, y_pred_rf),
        'mae': mean_absolute_error(y_test, y_pred_rf),
        'predictions': y_pred_rf
    }
    print(f"Лучшие параметры: {rf_grid.best_params_}")
    print(f"R^2: {results['random_forest']['r2']:.4f}")
    
    return results

In [27]:
# Обучаем baseline модели
print(f"\n{GREEN}{'='*50}{RESET}")
print(f"{GREEN}Запуск обучения baseline моделей{RESET}")
print(f"{GREEN}{'='*50}{RESET}\n")

with mlflow.start_run(run_name="baseline_models_tuned", nested=True):
    baseline_results = train_baseline_with_tuning(X_train, y_train, X_test, y_test)
    
    # Логируем результаты
    for name, metrics in baseline_results.items():
        mlflow.log_metrics({
            f"{name}_r2": metrics['r2'],
            f"{name}_mae": metrics['mae']
        })
        mlflow.log_params({f"{name}_{k}": v for k, v in metrics['best_params'].items()})


Запуск обучения baseline моделей

Обучаем Ridge Regression с подбором параметров...
Лучшие параметры: {'alpha': 100.0}
R^2: 0.3747

Обучаем Random Forest с подбором параметров...
Лучшие параметры: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}
R^2: 0.9040
🏃 View run baseline_models_tuned at: http://localhost:5050/#/experiments/1/runs/35c6671a1d694b998026a41121c6d606
🧪 View experiment at: http://localhost:5050/#/experiments/1


## 5. Catboost

In [28]:
def train_catboost(X_train, y_train, X_val, y_val, X_test, y_test):

    print(f"\n{BLUE}Обучаем CatBoost с оптимизацией гиперпараметров...{RESET}")

    def objective(trial):
        params = {
            'iterations': trial.suggest_int('iterations', 100, 500),
            'depth': trial.suggest_int('depth', 4, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
            'random_seed': 42,
            'verbose': False
        }

        model = CatBoostRegressor(**params)
        model.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=False)
        return r2_score(y_val, model.predict(X_val))

    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=20, show_progress_bar=False)

    best_params = study.best_params
    catboost_model = CatBoostRegressor(**best_params, random_seed=42, verbose=False)
    catboost_model.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=False)

    y_pred_test = catboost_model.predict(X_test)
    y_pred_train = catboost_model.predict(X_train)
    y_pred_val = catboost_model.predict(X_val)

    results = {
        'model': catboost_model,
        'best_params': best_params,
        'r2': r2_score(y_test, y_pred_test),
        'mae': mean_absolute_error(y_test, y_pred_test),
        'train_r2': r2_score(y_train, y_pred_train),
        'train_mae': mean_absolute_error(y_train, y_pred_train),
        'val_r2': r2_score(y_val, y_pred_val),
        'val_mae': mean_absolute_error(y_val, y_pred_val),
        'predictions': y_pred_test
    }

    print(f"Лучшие параметры: iterations={best_params['iterations']}, depth={best_params['depth']}, lr={best_params['learning_rate']:.3f}")
    print(f"R^2: {results['r2']:.4f}")

    return results

In [29]:
mlflow.end_run()  # завершаем незакрытый ран если есть

experiment_name = "gradient_boosting_optimized"
artifact_location = "s3://mlflow-bucket/mlflow"

exp = mlflow.get_experiment_by_name(experiment_name)
if exp is None:
    exp_id = MlflowClient().create_experiment(name=experiment_name, artifact_location=artifact_location)
else:
    exp_id = exp.experiment_id

mlflow.set_experiment(experiment_name)
registered_model_name = "catboost_optimized"

with mlflow.start_run(run_name="catboost_optimized_final", experiment_id=exp_id):
    catboost_results = train_catboost(X_train, y_train, X_val, y_val, X_test, y_test)

    mlflow.log_metrics({
        "test_r2": catboost_results['r2'],
        "test_mae": catboost_results['mae'],
        "train_r2": catboost_results['train_r2'],
        "train_mae": catboost_results['train_mae'],
        "val_r2": catboost_results['val_r2'],
        "val_mae": catboost_results['val_mae'],
    })
    mlflow.log_params(catboost_results['best_params'])

    signature = infer_signature(X_train, catboost_results['model'].predict(X_train))
    model_info = mlflow.sklearn.log_model(
        sk_model=catboost_results['model'],
        artifact_path="model",
        signature=signature,
        input_example=X_train.head(5),
        registered_model_name=registered_model_name,
    )

    client = MlflowClient()
    client.set_model_version_tag(registered_model_name, model_info.registered_model_version, "env", "PRD")
    client.set_registered_model_alias(registered_model_name, "prd", model_info.registered_model_version)

    print(f"Модель зарегистрирована: {registered_model_name} v{model_info.registered_model_version}")
    print("Alias 'prd' и тег PRD добавлены")

🏃 View run auspicious-stag-273 at: http://localhost:5050/#/experiments/1/runs/df45faa44fe84696a3e77bac76090ae9
🧪 View experiment at: http://localhost:5050/#/experiments/1


[I 2026-06-03 09:32:21,466] A new study created in memory with name: no-name-d1b6f0c1-ae50-4ec5-b260-a2f93c989020



Обучаем CatBoost с оптимизацией гиперпараметров...


[I 2026-06-03 09:32:23,800] Trial 0 finished with value: 0.7373202063019377 and parameters: {'iterations': 250, 'depth': 10, 'learning_rate': 0.1205712628744377, 'l2_leaf_reg': 6.387926357773329}. Best is trial 0 with value: 0.7373202063019377.
[I 2026-06-03 09:32:23,997] Trial 1 finished with value: 0.6529755201682498 and parameters: {'iterations': 162, 'depth': 5, 'learning_rate': 0.012184186502221764, 'l2_leaf_reg': 8.795585311974417}. Best is trial 0 with value: 0.7373202063019377.
[I 2026-06-03 09:32:24,910] Trial 2 finished with value: 0.6974761553112832 and parameters: {'iterations': 341, 'depth': 8, 'learning_rate': 0.010725209743171996, 'l2_leaf_reg': 9.72918866945795}. Best is trial 0 with value: 0.7373202063019377.
[I 2026-06-03 09:32:25,373] Trial 3 finished with value: 0.7109139353768177 and parameters: {'iterations': 433, 'depth': 5, 'learning_rate': 0.01855998084649059, 'l2_leaf_reg': 2.650640588680904}. Best is trial 0 with value: 0.7373202063019377.
[I 2026-06-03 09:32

Лучшие параметры: iterations=497, depth=10, lr=0.065
R^2: 0.9207


Registered model 'catboost_optimized' already exists. Creating a new version of this model...
2026/06/03 09:32:55 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: catboost_optimized, version 2


Модель зарегистрирована: catboost_optimized v2
Alias 'prd' и тег PRD добавлены
🏃 View run catboost_optimized_final at: http://localhost:5050/#/experiments/1/runs/7259ab0e188f478898b3d07a06366d76
🧪 View experiment at: http://localhost:5050/#/experiments/1


Created version '2' of model 'catboost_optimized'.


## 6. Gradient Boosting

In [30]:
def train_gradient_boosting_optuna(X_train, y_train, X_val, y_val, X_test, y_test):

    print(f"\n{BLUE}Обучаем Gradient Boosting с оптимизацией гиперпараметров...{RESET}")
    
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 500),
            'max_depth': trial.suggest_int('max_depth', 3, 8),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
            'subsample': trial.suggest_float('subsample', 0.7, 1.0),
            'random_state': 42
        }
        
        model = GradientBoostingRegressor(**params)
        model.fit(X_train, y_train)
        return r2_score(y_val, model.predict(X_val))
    
    # Оптимизация
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=30, show_progress_bar=False)
    
    # Финальная модель
    best_params = study.best_params
    gbr_optimized = GradientBoostingRegressor(**best_params, random_state=42)
    gbr_optimized.fit(X_train, y_train)
    
    y_pred_gbr = gbr_optimized.predict(X_test)
    y_pred_train = gbr_optimized.predict(X_train)
    y_pred_val = gbr_optimized.predict(X_val)
    
    results = {
        'model': gbr_optimized,
        'best_params': best_params,
        'r2': r2_score(y_test, y_pred_gbr),
        'mae': mean_absolute_error(y_test, y_pred_gbr),
        'mse': mean_squared_error(y_test, y_pred_gbr),
        'train_r2': r2_score(y_train, y_pred_train),
        'train_mae': mean_absolute_error(y_train, y_pred_train),
        'val_r2': r2_score(y_val, y_pred_val),
        'val_mae': mean_absolute_error(y_val, y_pred_val),
        'predictions': y_pred_gbr
    }
    
    print(f"Лучшие параметры:")
    for k, v in best_params.items():
        print(f"{k}: {v}")
    print(f"R^2: {results['r2']:.4f}")
    
    return results

In [31]:
exp_gbr = mlflow.get_experiment_by_name("gradient_boosting_optimized")
exp_id_gbr = exp_gbr.experiment_id if exp_gbr else None

with mlflow.start_run(run_name="gradient_boosting_comparison", experiment_id=exp_id_gbr):
    gbr_results = train_gradient_boosting_optuna(X_train, y_train, X_val, y_val, X_test, y_test)

    mlflow.log_params(gbr_results['best_params'])
    mlflow.log_metrics({
        "test_r2": gbr_results['r2'],
        "test_mae": gbr_results['mae'],
        "test_mse": gbr_results['mse'],
        "train_r2": gbr_results['train_r2'],
        "train_mae": gbr_results['train_mae'],
        "val_r2": gbr_results['val_r2'],
        "val_mae": gbr_results['val_mae'],
    })

    print(f"GBR залоггирован для сравнения (не PRD): R2={gbr_results['r2']:.4f}")


[I 2026-06-03 09:32:55,877] A new study created in memory with name: no-name-aa993312-f42d-42ff-8f9b-bbe508e10463



Обучаем Gradient Boosting с оптимизацией гиперпараметров...


[I 2026-06-03 09:32:58,744] Trial 0 finished with value: 0.7193660590728239 and parameters: {'n_estimators': 250, 'max_depth': 8, 'learning_rate': 0.08960785365368121, 'min_samples_split': 7, 'min_samples_leaf': 1, 'subsample': 0.7467983561008608}. Best is trial 0 with value: 0.7193660590728239.
[I 2026-06-03 09:33:00,545] Trial 1 finished with value: 0.7283345367231266 and parameters: {'n_estimators': 123, 'max_depth': 8, 'learning_rate': 0.06054365855469246, 'min_samples_split': 8, 'min_samples_leaf': 1, 'subsample': 0.9909729556485982}. Best is trial 1 with value: 0.7283345367231266.
[I 2026-06-03 09:33:03,615] Trial 2 finished with value: 0.7215983285866014 and parameters: {'n_estimators': 433, 'max_depth': 4, 'learning_rate': 0.017240892195821537, 'min_samples_split': 3, 'min_samples_leaf': 2, 'subsample': 0.8574269294896714}. Best is trial 1 with value: 0.7283345367231266.
[I 2026-06-03 09:33:05,429] Trial 3 finished with value: 0.7302466321896628 and parameters: {'n_estimators':

Лучшие параметры:
n_estimators: 406
max_depth: 8
learning_rate: 0.01757255779054591
min_samples_split: 6
min_samples_leaf: 2
subsample: 0.7661022227562746
R^2: 0.9106
GBR залоггирован для сравнения (не PRD): R2=0.9106
🏃 View run gradient_boosting_comparison at: http://localhost:5050/#/experiments/1/runs/e5028834733249559ad12de7f935b3f7
🧪 View experiment at: http://localhost:5050/#/experiments/1


## 7. Сравнение моделей

In [32]:
all_models = {
    'Ridge Regression': baseline_results['ridge'],
    'Random Forest': baseline_results['random_forest'],
    'CatBoost': catboost_results,
    'Gradient Boosting': gbr_results
}

print(f"\n{GREEN}{'='*60}{RESET}")
print(f"{GREEN}СРАВНЕНИЕ МОДЕЛЕЙ{RESET}")
print(f"{GREEN}{'='*60}{RESET}\n")

comparison_df = pd.DataFrame([{
    'Модель': name,
    'R^2': metrics['r2'],
    'MAE': metrics['mae'],
    'Качество': 'Отлично' if metrics['r2'] > 0.95 else 'Хорошо' if metrics['r2'] > 0.9 else 'Средне'
} for name, metrics in all_models.items()])

comparison_df = comparison_df.sort_values('R^2', ascending=False)
print(comparison_df.to_string(index=False))

print(f"\n{BLUE}ЛУЧШАЯ МОДЕЛЬ:{RESET}")
best_model_name = comparison_df.iloc[0]['Модель']
best_r2 = comparison_df.iloc[0]['R^2']
print(f"  {best_model_name} с R^2 = {best_r2:.4f}")

print(f"\n{BLUE}СРАВНИТЕЛЬНЫЙ АНАЛИЗ:{RESET}")

# Сравнение с Ridge
improvement_vs_ridge = (gbr_results['r2'] - baseline_results['ridge']['r2']) * 100
print(f"Gradient Boosting лучше Ridge Regression на {improvement_vs_ridge:+.1f}% по R²")

# Сравнение с Random Forest
improvement_vs_rf = (gbr_results['r2'] - baseline_results['random_forest']['r2']) * 100
print(f"Gradient Boosting лучше Random Forest на {improvement_vs_rf:+.1f}% по R^2")

# Сравнение с CatBoost
improvement_vs_catboost = (gbr_results['r2'] - catboost_results['r2']) * 100
if improvement_vs_catboost > 0:
    print(f"Gradient Boosting лучше CatBoost на {improvement_vs_catboost:+.1f}% по R^2")
else:
    print(f"CatBoost лучше Gradient Boosting на {abs(improvement_vs_catboost):+.1f}% по R^2")

print(f"\n{BLUE}🎯 ВЫВОД:{RESET}")
if best_model_name == "Gradient Boosting":
    print(f"Gradient Boosting показал наилучший результат на тестовых данных.")
    print(f"Модель объясняет {best_r2*100:.1f}% дисперсии целевой переменной.")
    print(f"Средняя абсолютная ошибка (MAE) составляет {gbr_results['mae']:.4f}.")
else:
    print(f"{best_model_name} показал наилучший результат на тестовых данных.")


СРАВНЕНИЕ МОДЕЛЕЙ

           Модель      R^2      MAE Качество
         CatBoost 0.920703 0.015417   Хорошо
Gradient Boosting 0.910611 0.015976   Хорошо
    Random Forest 0.903995 0.016907   Хорошо
 Ridge Regression 0.374661 0.048315   Средне

ЛУЧШАЯ МОДЕЛЬ:
  CatBoost с R^2 = 0.9207

СРАВНИТЕЛЬНЫЙ АНАЛИЗ:
Gradient Boosting лучше Ridge Regression на +53.6% по R²
Gradient Boosting лучше Random Forest на +0.7% по R^2
CatBoost лучше Gradient Boosting на +1.0% по R^2

🎯 ВЫВОД:
CatBoost показал наилучший результат на тестовых данных.


## 8. Анализ ошибок лучшей модели

In [33]:
# Берем лучшую модель
best_model = all_models[best_model_name]['model']
best_predictions = all_models[best_model_name]['predictions']

error_df = pd.DataFrame({
    'actual': y_test.values,
    'predicted': best_predictions,
    'error': y_test.values - best_predictions,
    'abs_error': np.abs(y_test.values - best_predictions)
})

top_errors = error_df.nlargest(20, 'abs_error')
top_errors.to_csv("top_20_errors.csv", index=False)
mlflow.log_artifact("top_20_errors.csv")

print(f"\n{BLUE}ТОП-20 ОШИБОК (модель: {best_model_name}):{RESET}")
print(top_errors[['actual', 'predicted', 'error']].to_string())

under = top_errors[top_errors['error'] > 0]
over = top_errors[top_errors['error'] < 0]

print(f"\nКАТЕГОРИЗАЦИЯ ОШИБОК:")
print(f"Заниженные предсказания (модель недооценила): {len(under)} шт")
print(f"Завышенные предсказания (модель переоценила): {len(over)} шт")

print(f"\n{BLUE}ДЕТАЛЬНЫЙ АНАЛИЗ ПРИЧИН ОШИБОК:{RESET}")

# --- 1. Объекты с одинаковыми предсказаниями ---
pred_counts = pd.Series(best_predictions).value_counts()
repeated_preds = pred_counts[pred_counts > 1]
if not repeated_preds.empty:
    print(f"\n1. ОБЪЕКТЫ С ОДИНАКОВЫМИ ПРЕДСКАЗАНИЯМИ:")
    for pred_val, cnt in repeated_preds.head(5).items():
        print(f"   predicted={pred_val:.6f} — встречается у {cnt} объектов")
    print("   Причина: у этих банкоматов схожий набор географических признаков")
    print("   (одинаковая инфраструктура вокруг). Gradient Boosting строит дерево решений")
    print("   и при попадании в один лист дерева выдает одинаковый прогноз.")
    print("   Корректировка невозможна без дополнительных признаков, разделяющих эти объекты.")

# --- 2. Ошибки на экстремальных значениях ---
q_high = error_df['actual'].quantile(0.90)
q_low = error_df['actual'].quantile(0.10)
extreme_mask = (error_df['actual'] > q_high) | (error_df['actual'] < q_low)
extreme_errors = error_df[extreme_mask]
print(f"\n2. ОШИБКИ НА ЭКСТРЕМАЛЬНЫХ ЗНАЧЕНИЯХ ТАРГЕТА:")
print(f"   Объектов с actual > {q_high:.3f} или < {q_low:.3f}: {extreme_mask.sum()} шт")
print(f"   Средняя abs_error на экстремальных: {extreme_errors['abs_error'].mean():.4f}")
print(f"   Средняя abs_error на остальных:     {error_df[~extreme_mask]['abs_error'].mean():.4f}")
print("   Причина: редкие объекты с нетипичной популярностью — модель не имеет")
print("   достаточно похожих примеров в обучающей выборке и регрессирует к среднему.")
print("   Корректировка невозможна без сбора дополнительных данных по редким локациям.")

# --- 3. Ошибки при таргете близком к нулю ---
near_zero_mask = error_df['actual'].abs() < 0.01
near_zero_errors = error_df[near_zero_mask]
print(f"\n3. ОШИБКИ ПРИ ТАРГЕТЕ БЛИЗКОМ К НУЛЮ:")
print(f"   Объектов с |actual| < 0.01: {near_zero_mask.sum()} шт")
print(f"   Средняя abs_error: {near_zero_errors['abs_error'].mean():.4f}")
print("   Причина: нейтральные локации (популярность ≈ 0) по признакам неотличимы")
print("   от слабо отрицательных — инфраструктура не даёт достаточного сигнала.")
print("   Корректировка невозможна в рамках текущего набора признаков.")

# --- Итог ---
print(f"\n{BLUE}ИТОГ АНАЛИЗА ОШИБОК:{RESET}")
print(f"  Средняя ошибка (MAE): {error_df['abs_error'].mean():.4f}")
print(f"  Максимальная ошибка:  {error_df['abs_error'].max():.4f}")
print(f"  95-й перцентиль:      {error_df['abs_error'].quantile(0.95):.4f}")
print("  Модель стабильно работает на 95% данных — ошибки сосредоточены")
print("  на редких экстремальных объектах и локациях со схожей инфраструктурой.")



ТОП-20 ОШИБОК (модель: CatBoost):
        actual  predicted     error
805  -0.000771  -0.097040  0.096270
553   0.170734   0.087679  0.083055
2392  0.177478   0.098800  0.078678
2484  0.177478   0.098800  0.078678
440   0.192010   0.127484  0.064526
1646  0.192010   0.127484  0.064526
929   0.029465  -0.034676  0.064141
2360 -0.048613   0.015091 -0.063704
941  -0.012571  -0.075548  0.062976
1032  0.002210  -0.059868  0.062078
2213  0.086121   0.147368 -0.061248
1270 -0.053980   0.007089 -0.061069
1777 -0.053980   0.007089 -0.061069
125  -0.035578   0.023136 -0.058714
98    0.009316   0.067724 -0.058408
2006  0.168546   0.110316  0.058229
1835 -0.021648  -0.079705  0.058057
1884 -0.039498  -0.097522  0.058024
928   0.169059   0.111355  0.057703
1233  0.169059   0.111355  0.057703

КАТЕГОРИЗАЦИЯ ОШИБОК:
Заниженные предсказания (модель недооценила): 14 шт
Завышенные предсказания (модель переоценила): 6 шт

ДЕТАЛЬНЫЙ АНАЛИЗ ПРИЧИН ОШИБОК:

1. ОБЪЕКТЫ С ОДИНАКОВЫМИ ПРЕДСКАЗАНИЯМИ:
   predi

## 9. Robustness тест лучшей модели

In [34]:
with mlflow.start_run(run_name="robustness_best_model", nested=True):
    noise_levels = [0.01, 0.05, 0.1]
    X_sample = X_test.iloc[:100]
    y_sample = y_test.iloc[:100]

    original_pred = best_model.predict(X_sample)
    original_mse = mean_squared_error(y_sample, original_pred)
    mlflow.log_metric("baseline_mse", original_mse)

    print(f"\n{BLUE}ROBUSTNESS TEST ({best_model_name}):{RESET}")

    change_pcts = []
    for noise in noise_levels:
        X_noisy = X_sample.copy()
        numeric_cols = X_noisy.select_dtypes(include=[np.number]).columns

        for col in numeric_cols:
            noise_vals = np.random.normal(0, noise * X_noisy[col].std(), len(X_noisy))
            X_noisy[col] += noise_vals

        noisy_pred = best_model.predict(X_noisy)
        noisy_mse = mean_squared_error(y_sample, noisy_pred)
        change_pct = ((noisy_mse - original_mse) / original_mse) * 100
        change_pcts.append(change_pct)

        mlflow.log_metric(f"mse_noise_{int(noise*100)}", noisy_mse)
        mlflow.log_metric(f"mse_change_{int(noise*100)}", change_pct)

        status = "устойчива" if abs(change_pct) < 10 else "чувствительна"
        print(f"  Noise {int(noise*100):2d}%: изменение MSE = {change_pct:+6.1f}% -> {status}")

    print(f"\n{BLUE}ТАКИМ ОБРАЗОМ:{RESET}")
    if all(abs(cp) < 10 for cp in change_pcts):
        print("Модель демонстрирует хорошую устойчивость к шуму до 10%")
    elif abs(change_pcts[0]) < 10:
        print("Модель устойчива к малому шуму (1-5%), но чувствительна к сильному шуму (>10%)")
    else:
        print("Модель чувствительна к шуму на всех уровнях")


ROBUSTNESS TEST (CatBoost):
  Noise  1%: изменение MSE =   +0.4% -> устойчива
  Noise  5%: изменение MSE =   -2.4% -> устойчива
  Noise 10%: изменение MSE =  +33.4% -> чувствительна

ТАКИМ ОБРАЗОМ:
Модель устойчива к малому шуму (1-5%), но чувствительна к сильному шуму (>10%)
🏃 View run robustness_best_model at: http://localhost:5050/#/experiments/1/runs/aa7a0ef82d474e17b3766e611a5c1b9b
🧪 View experiment at: http://localhost:5050/#/experiments/1


## 10.  Learning Curve

In [35]:
train_sizes, train_scores, val_scores = learning_curve(
    best_model, X_train, y_train, cv=3, 
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='r2', n_jobs=-1
)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Training', linewidth=2)
plt.plot(train_sizes, val_scores.mean(axis=1), 'o-', label='Validation', linewidth=2)
plt.xlabel('Training examples', fontsize=12)
plt.ylabel('R^2 Score', fontsize=12)
plt.title(f'Learning Curves - {best_model_name}', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('learning_curves.png', dpi=150)
mlflow.log_artifact('learning_curves.png')
plt.close()

print("Learning curve сохранена")

Learning curve сохранена


### Интерпретация

- Кривые обучения **сходятся** -> модель не переобучена
- Разрыв между train и validation **минимальный** -> хорошая обобщающая способность
- С ростом объема данных качество **стабилизируется** -> дальнейшее увеличение выборки не даст значительного улучшения
"""


## 11. Проверка модели из реестра (PRD)

In [36]:
loaded_model = mlflow.pyfunc.load_model(f"models:/catboost_optimized@prd")

test_sample = X_test.head(3).copy()
int_cols = ['schools_nearby', 'supermarket_nearby', 'mall_nearby', 'bar_nearby',
            'cafe_nearby', 'restaurant_nearby', 'police_nearby', 'post_office_nearby',
            'place_of_worship_nearby', 'university_nearby', 'cinema_nearby',
            'casino_nearby', 'nightclub_nearby']
float_cols = ['atm_group', 'lat', 'long', 'atm_nearby', 'population']

test_sample[int_cols] = test_sample[int_cols].astype('int64')
test_sample[float_cols] = test_sample[float_cols].astype('float64')

sample_pred = loaded_model.predict(test_sample)

print("Модель успешно загружена из MLflow Registry (catboost_optimized@prd)")
print("\nПредсказания на первых 3 примерах:")
for i, pred in enumerate(sample_pred):
    print(f"  Sample {i+1}: {pred:.6f}")


Модель успешно загружена из MLflow Registry (catboost_optimized@prd)

Предсказания на первых 3 примерах:
  Sample 1: -0.057810
  Sample 2: -0.024707
  Sample 3: -0.025409


## Выводы

### Основные результаты

| Показатель | Значение |
|-----------|----------|
| **Лучшая модель** | CatBoost |
| **R² на тесте** | 0.9207 |
| **MAE на тесте** | 0.0154 |

### Ключевые выводы

1. **CatBoost с оптимизацией гиперпараметров показал наилучший результат** среди всех рассмотренных моделей (R²=0.9207 vs 0.9106 у GBR)
2. **Модель устойчива к шуму до 10%** — изменение MSE не превышает 10% даже при сильном шуме
3. **Основные ошибки связаны с занижением предсказаний** (14 из 20) на объектах с экстремальными значениями таргета
4. **Все эксперименты сохранены в MLflow** с возможностью воспроизведения, модель зарегистрирована как `catboost_optimized` с тегом `PRD`

## Проверка результатов в MLflow

In [37]:
runs_df = mlflow.search_runs(
    experiment_names=[experiment_name],
    order_by=["metrics.test_r2 DESC"],
)

print(runs_df[["run_id", "metrics.test_r2", "metrics.test_mae", "artifact_uri"]].head())

# Показать запуски
print(mlflow.search_runs(experiment_names=[experiment_name]))

                             run_id  metrics.test_r2  metrics.test_mae  \
0  7259ab0e188f478898b3d07a06366d76         0.920703          0.015417   
1  4cfda1d4ffd34b93a90ea04bf876c334         0.920703          0.015417   
2  e5028834733249559ad12de7f935b3f7         0.910611          0.015976   
3  114b937734784991a6499e8a74704d78         0.910611          0.015976   
4  7d6ff9434d314626b8d6f496c4d7ffec         0.910611          0.015976   

                                        artifact_uri  
0  s3://mlflow-bucket/mlflow/7259ab0e188f478898b3...  
1  s3://mlflow-bucket/mlflow/4cfda1d4ffd34b93a90e...  
2  s3://mlflow-bucket/mlflow/e5028834733249559ad1...  
3  s3://mlflow-bucket/mlflow/114b937734784991a649...  
4  s3://mlflow-bucket/mlflow/7d6ff9434d314626b8d6...  
                              run_id experiment_id    status  \
0   aa7a0ef82d474e17b3766e611a5c1b9b             1  FINISHED   
1   bc02d69f2d4b4a6cacef80cae1588973             1   RUNNING   
2   e5028834733249559ad12de7f935b